#### Task 2: Read purchase_data.csv and run a one-way ANOVA on purchase_value across age groups.
Run task1_create_purchase_data.py first so purchase_data.csv exists.

In [2]:
import os
import pandas as pd
from scipy import stats
 
HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
df = pd.read_csv(os.path.join(HERE, "purchase_data.csv"))
 
print("H0: mean purchase_value is the same in all three age groups")
print("H1: at least one age group has a different mean purchase_value\n")
 
summary = df.groupby("age_group")["purchase_value"].agg(["count", "mean", "std"]).round(2)
print(summary.to_string(), "\n")
 
groups = [g["purchase_value"].values for _, g in df.groupby("age_group")]
f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic = {f_stat:.4f}")
print(f"p-value     = {p_value:.6f}")
 
alpha = 0.05
print(f"\nAt alpha = {alpha}: " + ("p < alpha -> reject H0 (significant difference between groups)."
                                   if p_value < alpha else "p >= alpha -> fail to reject H0."))
 
# Assumption checks (informal with only 5 values per group)
print("\nAssumption checks:")
for name, g in df.groupby("age_group"):
    print(f"  Shapiro-Wilk normality, {name}: p = {stats.shapiro(g['purchase_value']).pvalue:.3f}")
print(f"  Levene equal-variance test: p = {stats.levene(*groups).pvalue:.3f}")
 
# Which groups differ? Tukey HSD post-hoc test
tukey = stats.tukey_hsd(*groups)
labels = sorted(df["age_group"].unique())
print("\nTukey HSD pairwise comparisons:")
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        diff = groups[i].mean() - groups[j].mean()
        print(f"  {labels[i]} vs {labels[j]}: mean difference = {diff:7.1f}, p = {tukey.pvalue[i][j]:.4f}")

H0: mean purchase_value is the same in all three age groups
H1: at least one age group has a different mean purchase_value

           count   mean    std
age_group                     
18-25          5  520.0  61.24
26-35          5  708.0  47.64
36-50          5  658.0  48.17 

F-statistic = 17.0528
p-value     = 0.000311

At alpha = 0.05: p < alpha -> reject H0 (significant difference between groups).

Assumption checks:
  Shapiro-Wilk normality, 18-25: p = 0.896
  Shapiro-Wilk normality, 26-35: p = 0.847
  Shapiro-Wilk normality, 36-50: p = 0.992
  Levene equal-variance test: p = 0.830

Tukey HSD pairwise comparisons:
  18-25 vs 26-35: mean difference =  -188.0, p = 0.0003
  18-25 vs 36-50: mean difference =  -138.0, p = 0.0036
  26-35 vs 36-50: mean difference =    50.0, p = 0.3256
